In [ ]:
# 1. Install PySpark in Colab (The Big Data Engine)
!pip install pyspark

In [ ]:
!pip install pyspark nltk

In [ ]:
import nltk
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, when, expr, udf
from pyspark.sql.types import ArrayType, StringType
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# 1. Download NLTK data for Lemmatization
print("Downloading NLTK Lemmatizer data...")
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer

print("Starting PySpark Engine...")
spark = SparkSession.builder.appName("UltimatePreProcessing").getOrCreate()

# 2. Load the raw data (with safeguards)
print("Loading raw data...")
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .csv("grab_reviews_raw.csv") # Change filename if necessary

df_clean = df.withColumn("score", expr("TRY_CAST(score AS INT)"))
df_clean = df_clean.dropna(subset=["content", "score"])

# --- Deduplication ---
print("Removing duplicate spam reviews...")
initial_count = df_clean.count()
df_clean = df_clean.dropDuplicates(["content"])
print(f"Dropped {initial_count - df_clean.count()} duplicate rows.")

# --- Remove Noise (URLs, Mentions, Lowercase) ---
print("Cleaning text (Lowercase, URLs, Mentions, Punctuation)...")
df_clean = df_clean.withColumn("content", lower(col("content")))
# Remove URLs (http://... or www...)
df_clean = df_clean.withColumn("content", regexp_replace(col("content"), r"http\S+|www\S+|https\S+", ""))
# Remove Mentions (@username)
df_clean = df_clean.withColumn("content", regexp_replace(col("content"), r"@\w+", ""))
# Remove everything except letters and spaces
df_clean = df_clean.withColumn("content", regexp_replace(col("content"), "[^a-z\\s]", ""))

# --- Tokenization ---
print("Applying Tokenization...")
tokenizer = Tokenizer(inputCol="content", outputCol="words")
df_clean = tokenizer.transform(df_clean)

# --- Stopword Removal (English + Malay) ---
print("Removing English and Malay Stopwords...")
english_stopwords = StopWordsRemover.loadDefaultStopWords("english")
malay_stopwords = ["dan", "itu", "ini", "yang", "di", "ke", "dari", "untuk", "dengan", "pada", "grab", "pun", "lah", "kat", "ni", "tu", "yg", "je", "macam"]
all_stopwords = english_stopwords + malay_stopwords

remover = StopWordsRemover(inputCol="words", outputCol="filtered_words", stopWords=all_stopwords)
df_clean = remover.transform(df_clean)

# --- Lemmatization ---
print("Applying Lemmatization (NLTK)...")
lemmatizer = WordNetLemmatizer()

# Create a Custom PySpark Function (UDF) to use NLTK inside Spark
def lemmatize_words(word_list):
    if word_list is None: return []
    return [lemmatizer.lemmatize(word) for word in word_list]

lemmatize_udf = udf(lemmatize_words, ArrayType(StringType()))
df_clean = df_clean.withColumn("lemmatized_words", lemmatize_udf(col("filtered_words")))

# Rejoin the array of words back into a normal sentence so it saves cleanly to CSV
rejoin_udf = udf(lambda words: " ".join(words) if words else "", StringType())
df_clean = df_clean.withColumn("final_clean_text", rejoin_udf(col("lemmatized_words")))

# ---  SENTIMENT RULES ---
print("Applying Sentiment Labels...")
df_clean = df_clean.withColumn("sentiment_category",
    when(col("score") >= 4, "Positive")
    .when(col("score") <= 2, "Negative")
    .otherwise("Neutral")
)
df_clean = df_clean.withColumn("sentiment_label",
    when(col("score") >= 4, 2.0)
    .when(col("score") <= 2, 0.0)
    .otherwise(1.0)
)

# REORDER AND CLEAN UP COLUMNS
front_columns = ["review_id", "username", "timestamp", "final_clean_text", "score", "sentiment_category", "sentiment_label"]
remaining_columns = [c for c in df_clean.columns if c not in front_columns and c not in ["content", "words", "filtered_words", "lemmatized_words"]]
df_clean = df_clean.select(*(front_columns + remaining_columns))

# --- Class Distribution Check ---
print("\n📊 --- Sentiment Class Distribution ---")
df_clean.groupBy("sentiment_category").count().show()

print("\n✨ --- Final Cleaned Data Preview ---")
df_clean.select("final_clean_text", "sentiment_category").show(5, truncate=60)

# Save the final file
csv_filename = "cleaned_data.csv"
df_clean.toPandas().to_csv(csv_filename, index=False)
print(f"\n✅ Success! Saved as '{csv_filename}'.")

# Auto-download
from google.colab import files
files.download(csv_filename)

Starting PySpark Engine...
Loading raw data...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Removing duplicate spam reviews...
Dropped -2220 duplicate rows.
Cleaning text (Lowercase, URLs, Mentions, Punctuation)...
Applying Tokenization...
Removing English and Malay Stopwords...
Applying Lemmatization (NLTK)...
Applying Sentiment Labels...

📊 --- Sentiment Class Distribution ---
+------------------+-----+
|sentiment_category|count|
+------------------+-----+
|           Neutral| 2404|
|          Positive|23620|
|          Negative|25084|
+------------------+-----+


✨ --- Final Cleaned Data Preview ---
+------------------------------------------------------------+------------------+
|                                            final_clean_text|sentiment_category|
+------------------------------------------------------------+------------------+
|sangat mengecewa selama aku duduk tak pernah aku dengar k...|          Negative|
|                               many feature became confusing|           Neutral|
|                                         bad navigation  map|          

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>